# 02 · SS causal layer

The SS-level causal DAG (`temperature -> load`, confounded by the diurnal calendar),
a DoWhy backdoor effect estimate with placebo / random-common-cause / subset refuters,
and the causal-augmented vs correlational comparison across a (clearly-labelled,
injected) regime change. Reported honestly, including the negative sign.

In [ ]:
import os
os.environ["FORECAUS_OFFLINE"] = "1"   # run fully offline on the committed fixtures
import warnings; warnings.filterwarnings("ignore")
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np, pandas as pd
from pathlib import Path
# Resolve figures/ whether executed from the repo root or from notebooks/.
FIG = Path("notebooks/figures") if Path("notebooks").is_dir() else Path("figures")
FIG.mkdir(parents=True, exist_ok=True)
print("offline:", os.environ["FORECAUS_OFFLINE"], "| figures ->", FIG)


In [ ]:
from forecaus_grid_odeon.causal import build_ss_dag, draw_dag
fig, ax = plt.subplots(figsize=(7, 5))
draw_dag(build_ss_dag(), ax=ax)
fig.savefig(FIG / '02_ss_dag.png', dpi=130, bbox_inches='tight'); plt.close(fig)
print('saved', FIG / '02_ss_dag.png')

In [ ]:
from forecaus_grid_odeon.eval import run_ss_causal_effect
result, refute = run_ss_causal_effect(num_simulations=30)
lo, hi = result['ci']
print(f"temp_c -> load_kw effect: {result['effect']:+.3f} kW/degC   90% CI [{lo:+.3f}, {hi:+.3f}]")
print('backdoor adjustment set:', sorted(result['backdoor_set']))
print('\nRefuters:')
print(refute[['original_effect', 'new_effect', 'p_value', 'passed']].round(3).to_string())
assert refute['passed'].all(), 'a refuter failed'

In [ ]:
from forecaus_grid_odeon.eval import run_ss_break_experiment, make_break_figure
exp = run_ss_break_experiment()
print(exp['table'][['MAPE_normal', 'MAPE_break', 'MAPE_degradation']].round(3).to_string())
make_break_figure(exp, exp['break_start'], str(FIG / '02_ss_causal_break.png'), ylabel='load [kW]')
print('saved', FIG / '02_ss_causal_break.png')